# AgencyOS — Phase 3 GPU and storage preflight

This is a non-destructive check. It does not download models or datasets, call AI APIs, or train.

Keep this notebook private. Run with a GPU enabled. The scratch-storage canary is temporary and is deleted immediately.

A local write test does not prove persistence across sessions; restart the notebook and verify a private checkpoint location before Phase 3 sign-off.

In [ ]:
import os, platform, shutil, sys, tempfile
from pathlib import Path
print('AgencyOS Phase 3 preflight')
print('Python:', platform.python_version())
if sys.version_info < (3, 10): raise RuntimeError('Python 3.10 or newer is required.')
import torch
print('PyTorch:', torch.__version__, '| CUDA build:', torch.version.cuda)
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU unavailable. Enable GPU in notebook settings.')
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i); gib=p.total_memory/(1024**3)
    print(f'GPU {i}: {p.name} | {gib:.2f} GiB | capability {p.major}.{p.minor}')
    if gib < 12: raise RuntimeError(f'GPU {i} has less than 12 GiB VRAM.')
probe=torch.ones((1024,1024),dtype=torch.float16,device='cuda:0')
print('CUDA allocation/release: PASS | checksum',float(probe.sum().item()))
del probe; torch.cuda.synchronize(); torch.cuda.empty_cache()
try:
    import psutil
    m=psutil.virtual_memory(); print(f'RAM: {m.total/(1024**3):.2f} GiB total | {m.available/(1024**3):.2f} GiB available')
except ImportError: print('RAM details unavailable; psutil is optional.')
target=Path(os.environ.get('AGENCYOS_CHECKPOINT_DIR','/kaggle/working')); target.mkdir(parents=True,exist_ok=True)
d=shutil.disk_usage(target); print(f'Scratch path: {target} | free: {d.free/(1024**3):.2f} GiB')
with tempfile.NamedTemporaryFile(prefix='agencyos-check-',dir=target,delete=True) as f: f.write(b'phase3-check'); f.flush()
print('Temporary storage write/delete: PASS')
print('RESULT: PASS — local hardware and scratch checks completed.')
print('PERSISTENCE GATE: still pending. Restart session and verify private checkpoint read-back before sign-off.')

## Phase 2 — Run the base-model comparison

This step downloads the two public candidate model weights from Hugging Face and runs eight fixed agency tasks on the free GPU. It does not fine-tune a model and does not call a paid AI API. Internet must be enabled in Kaggle. Attach the repository snapshot (or the `benchmark_candidates.py` file) as a private notebook input first. The challenger may fail if the T4 runs out of memory; the script records that failure rather than silently switching to a paid service.

In [ ]:
import importlib.metadata, subprocess, sys
try:
    current_transformers = importlib.metadata.version('transformers')
except importlib.metadata.PackageNotFoundError:
    current_transformers = '0'
from packaging.version import Version
if Version(current_transformers) < Version('4.53.0'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'transformers>=4.53.0', 'accelerate>=1.0.0', 'safetensors>=0.4.5'])
else:
    print('Transformers already meets the Phase 2 minimum:', current_transformers)


In [ ]:
import sys, subprocess
from pathlib import Path
script_matches = list(Path("/kaggle/input").rglob("benchmark_candidates.py"))
if not script_matches:
    raise FileNotFoundError("Attach the AgencyOS repository snapshot or upload training/phase2/benchmark_candidates.py as a private Kaggle input.")
script = script_matches[0]
print("Benchmark script:", script)
subprocess.run([
    sys.executable, str(script),
    "--output-dir", "/kaggle/working/agencyos-phase2-benchmark",
    "--max-input-tokens", "1024",
    "--max-new-tokens", "192",
], check=True)
print("Benchmark outputs saved under /kaggle/working/agencyos-phase2-benchmark")


## Phase 3 — Create a private persistence canary

Run this cell after the preflight. Then choose **Save Version / Save & Run All** and keep the notebook private. After the version finishes, start a fresh session and attach the previous private notebook version's output. Only then run the read-back cell below. Do not treat the original `/kaggle/working` file as proof of persistence.

In [ ]:
from pathlib import Path
import hashlib, json, datetime
canary_dir = Path("/kaggle/working/agencyos-checkpoint-canary")
canary_dir.mkdir(parents=True, exist_ok=True)
canary_payload = "AGENCYOS_PRIVATE_CHECKPOINT_CANARY_V1"
(canary_dir / "canary.txt").write_text(canary_payload, encoding="utf-8")
canary_manifest = {
    "purpose": "persistence-only-canary",
    "sha256": hashlib.sha256(canary_payload.encode()).hexdigest(),
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
(canary_dir / "manifest.json").write_text(json.dumps(canary_manifest, indent=2), encoding="utf-8")
print("Canary created. Save this notebook version privately, then verify from a fresh session.")
print(json.dumps(canary_manifest, indent=2))


### Fresh-session read-back — run only after starting a new session

Attach the previous notebook version's private output as an input. This cell searches attached inputs, verifies the content and SHA-256, and fails closed if it cannot find the saved artifact.

In [ ]:
from pathlib import Path
import hashlib, json
matches = list(Path("/kaggle/input").rglob("canary.txt"))
verified = False
for canary_file in matches:
    manifest_file = canary_file.parent / "manifest.json"
    if not manifest_file.exists():
        continue
    payload = canary_file.read_text(encoding="utf-8")
    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
    actual_hash = hashlib.sha256(payload.encode()).hexdigest()
    if payload == "AGENCYOS_PRIVATE_CHECKPOINT_CANARY_V1" and actual_hash == manifest.get("sha256"):
        print("PERSISTENCE READ-BACK: PASS")
        print("Attached artifact:", canary_file)
        print("SHA-256:", actual_hash)
        verified = True
        break
if not verified:
    raise RuntimeError("PERSISTENCE READ-BACK: FAIL — prior private notebook output was not found or checksum did not match.")
